# Complete Guide: Synthetic Void Data Generation & Training

This tutorial walks you through **everything** in this repo, step by step.
No prior knowledge required — just follow along.

## What this repo does

It generates synthetic X-ray images with void defects for training segmentation models.
There are two approaches ("Epics"):

| Epic | Approach | When to use |
|------|----------|-------------|
| **Epic 1** | Physics-based (Beer-Lambert law) | Fast, no GPU needed, good baseline |
| **Epic 2** | ControlNet + Stable Diffusion | Photorealistic, needs GPU + real data |

## What you will learn

1. How to install and set up the environment
2. Epic 1: Test each function individually, then generate a full dataset
3. Epic 1: Use your own real images as backgrounds
4. Epic 2: Prepare crops, train ControlNet, generate, and paste
5. End-to-end: from raw images to a trained model

## Prerequisites

- Python 3.12
- For Epic 1: CPU is enough
- For Epic 2 training/generation: NVIDIA GPU with >= 8 GB VRAM

---
# Part 0: Installation & Setup

Run these commands in your terminal **once** to set up the environment:

```bash
# Clone and enter the repo
cd synthetic_data

# Install with uv (recommended)
uv sync

# For Epic 2 (ControlNet training/generation), also install:
uv pip install -e ".[epic2]"
```

Verify the installation:

In [ ]:
# This cell should run without errors
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Epic 1 imports
from udm_epic1 import SyntheticSampleGenerator, GeneratorConfig
from udm_epic1.physics.beer_lambert import BeerLambertSimulator, BeerLambertConfig
from udm_epic1.generators.void_shapes import VoidShapeGenerator, VoidGeometry
from udm_epic1.augmentation.transforms import AugmentationPipeline, AugmentationConfig

# Epic 2 imports
from udm_epic2 import CropConfig, extract_crops_for_pair, process_crop_dataset, edge_map_from_mask
from udm_epic2.integration.paste import paste_defect_on_background
from udm_epic2.quality.filter import laplacian_variance, passes_quality_gate

print("All imports OK")

---
# Part 1: Epic 1 — Physics-Based Void Synthesis

Epic 1 generates synthetic voids using the **Beer-Lambert law** of X-ray attenuation.
Voids are air pockets (no attenuation) so they appear **brighter** than surrounding solder.

The pipeline has 4 stages:
1. **Background** — generate or load a base X-ray image
2. **Void shapes** — create binary masks (ellipse, blob, elongated, cluster)
3. **Composition** — insert voids into the background using physics
4. **Augmentation** — apply transforms at training time (not saved to disk)

## 1.1 Testing the Physics Engine (`BeerLambertSimulator`)

The simulator generates realistic X-ray background fields and inserts voids.

In [ ]:
# Create a physics simulator with default settings
physics_cfg = BeerLambertConfig(
    mu_background_range=(0.8, 1.2),  # material attenuation variation
    mu_void=0.0,                      # air has no attenuation
    thickness_range=(0.3, 1.0),       # solder thickness variation
    apply_sft_correction=True,        # simulate Nordson MatriX SFT
    sft_sigma=15.0,
)
sim = BeerLambertSimulator(physics_cfg, rng=np.random.default_rng(42))

# Generate a synthetic background (no voids yet)
bg = sim.generate_background_field(height=256, width=256)

print(f"Background: shape={bg.shape}, dtype={bg.dtype}, range=[{bg.min():.3f}, {bg.max():.3f}]")

plt.figure(figsize=(5, 5))
plt.imshow(bg, cmap='gray')
plt.title('Synthetic X-ray background (Beer-Lambert)')
plt.colorbar(label='Normalized intensity')
plt.axis('off')
plt.show()

In [ ]:
# Now insert a void into the background
# First, create a simple circular void mask
void_mask = np.zeros((256, 256), dtype=np.uint8)
cv2.circle(void_mask, center=(128, 128), radius=30, color=255, thickness=-1)

# Insert using physics — void brightness = local bg + contrast boost
with_void = sim.insert_void(
    background=bg,
    void_mask=void_mask,
    contrast=0.2,       # how much brighter the void is
    edge_sigma=1.5,     # soft edge for realism
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(bg, cmap='gray'); axes[0].set_title('Background')
axes[1].imshow(void_mask, cmap='gray'); axes[1].set_title('Void mask')
axes[2].imshow(with_void, cmap='gray'); axes[2].set_title('After void insertion')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

# Verify: void region should be brighter
bg_mean = bg[void_mask > 127].mean()
void_mean = with_void[void_mask > 127].mean()
print(f"Background mean in void region: {bg_mean:.4f}")
print(f"After insertion mean:            {void_mean:.4f}")
print(f"Void is brighter: {void_mean > bg_mean}  (expected: True)")

In [ ]:
# Test normalization (p2/p98 percentile — same as what gets baked into ONNX)
normalized = sim.percentile_normalize(with_void)
uint16_img = sim.to_uint16(normalized)
uint8_img = sim.to_uint8(normalized)

print(f"Normalized: range=[{normalized.min():.3f}, {normalized.max():.3f}]")
print(f"uint16:     range=[{uint16_img.min()}, {uint16_img.max()}], dtype={uint16_img.dtype}")
print(f"uint8:      range=[{uint8_img.min()}, {uint8_img.max()}], dtype={uint8_img.dtype}")

## 1.2 Testing Void Shape Generation (`VoidShapeGenerator`)

There are 4 void morphologies, matching real-world IPC-7711 distribution:
- **Ellipse** (45%) — most common, trapped gas
- **Irregular blob** (35%) — coalesced voids
- **Elongated** (12%) — along pad edges
- **Cluster** (8%) — micro-voids in proximity

In [ ]:
shape_gen = VoidShapeGenerator(rng=np.random.default_rng(42))
H, W = 256, 256

# Generate one of each shape type
shapes = ['ellipse', 'irregular_blob', 'elongated', 'cluster']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, shape_name in zip(axes, shapes):
    geom = VoidGeometry(
        cx=128, cy=128,
        area_fraction=0.05,      # 5% of image area
        shape=shape_name,
        contrast=0.2,
        edge_sigma=1.0,
    )
    mask = shape_gen.generate(H, W, geom)
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'{shape_name}\n{(mask > 127).sum()} px')
    ax.axis('off')

plt.suptitle('4 Void Morphologies', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Test random placement with collision avoidance
geom = shape_gen.sample_geometry(
    height=256, width=256,
    min_area_fraction=0.005,
    max_area_fraction=0.08,
    shape_weights={'ellipse': 0.45, 'irregular_blob': 0.35, 'elongated': 0.12, 'cluster': 0.08},
    existing_masks=None,       # no existing voids to avoid
    min_separation=8,
)

print(f"Sampled geometry:")
print(f"  Shape:    {geom.shape}")
print(f"  Center:   ({geom.cx}, {geom.cy})")
print(f"  Area:     {geom.area_fraction:.4f} ({geom.area_fraction*100:.1f}% of image)")
print(f"  Contrast: {geom.contrast:.3f}")
print(f"  Edge blur: {geom.edge_sigma:.2f}")

## 1.3 Testing the Full Generator (`SyntheticSampleGenerator`)

This combines physics + shapes into one call that produces `(image, mask, metadata)`.

In [ ]:
cfg = GeneratorConfig(
    height=256, width=256,
    void_count_range=(1, 5),
    min_area_fraction=0.005,
    max_area_fraction=0.15,
    empty_image_fraction=0.0,  # no empty images for this demo
)
gen = SyntheticSampleGenerator(cfg, seed=42)

image, mask, meta = gen.generate(image_id='test_001', split='train')

print(f"Image: shape={image.shape}, dtype={image.dtype}")
print(f"Mask:  shape={mask.shape}, dtype={mask.dtype}, unique={np.unique(mask)}")
print(f"Metadata:")
print(f"  Voids placed:       {meta.n_voids}")
print(f"  Total void area:    {meta.total_void_area_fraction:.4f}")
print(f"  Has voids:          {meta.has_voids}")
print(f"  Background mu mean: {meta.background_mu_mean:.4f}")
print(f"  Void details:")
for i, v in enumerate(meta.void_geometries):
    print(f"    [{i}] {v['shape']:16s} area={v['area_fraction']:.4f}  contrast={v['contrast']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(image.astype(np.float32) / 65535.0, cmap='gray')
axes[0].set_title(f'Synthetic image ({meta.n_voids} voids)')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Ground truth mask')

# Overlay: mask on image
display_img = (image.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
overlay = cv2.cvtColor(display_img, cv2.COLOR_GRAY2BGR)
overlay[mask > 127] = [0, 0, 255]  # red overlay on voids
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title('Overlay (red = void)')

for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Test reproducibility — same seed = identical output
gen_a = SyntheticSampleGenerator(cfg, seed=42)
gen_b = SyntheticSampleGenerator(cfg, seed=42)

img_a, mask_a, _ = gen_a.generate('repro_test')
img_b, mask_b, _ = gen_b.generate('repro_test')

print(f"Images identical: {np.array_equal(img_a, img_b)}  (expected: True)")
print(f"Masks identical:  {np.array_equal(mask_a, mask_b)}  (expected: True)")

## 1.4 Using Your Own Real Images as Backgrounds

Instead of synthetic backgrounds, pass your real void-free image to `generate(background=...)`.
The generator handles resizing, normalization, and bit-depth conversion automatically.

In [ ]:
# --- Replace this block with your own image ---
# real_bg = cv2.imread("path/to/your/normal_image.png", cv2.IMREAD_UNCHANGED)

# For this demo, simulate a realistic-looking background
rng = np.random.default_rng(0)
real_bg = (0.3 + 0.05 * rng.standard_normal((512, 512))).astype(np.float32)
real_bg = np.clip(real_bg, 0.0, 1.0)

# Generate with real background
cfg_real = GeneratorConfig(height=512, width=512)
gen_real = SyntheticSampleGenerator(cfg_real, seed=42)

image_r, mask_r, meta_r = gen_real.generate(
    image_id='real_bg_001',
    background=real_bg,     # <--- pass your image here
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(real_bg, cmap='gray')
axes[0].set_title('Your original (no voids)')
axes[1].imshow(image_r.astype(np.float32) / 65535.0, cmap='gray')
axes[1].set_title(f'With synthetic voids ({meta_r.n_voids})')
axes[2].imshow(mask_r, cmap='gray')
axes[2].set_title('Ground truth mask')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 1.5 Testing Augmentation (`AugmentationPipeline`)

Augmentation is applied at **training time only** — augmented images are **never saved to disk**.
This avoids overfitting on augmented artifacts.

In [ ]:
aug_cfg = AugmentationConfig(
    horizontal_flip_prob=0.5,
    vertical_flip_prob=0.5,
    rotate_90_prob=0.3,
    brightness_limit=0.1,
    contrast_limit=0.1,
    brightness_contrast_prob=0.4,
    gaussian_blur_prob=0.2,
    ring_artifact_prob=0.3,       # simulate SAKI ring artifacts
    stitch_seam_prob=0.2,         # simulate stitching seams
)
aug = AugmentationPipeline(aug_cfg, rng=np.random.default_rng(0))

# Use a float32 version of our generated image
img_f32 = image.astype(np.float32) / 65535.0

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(img_f32, cmap='gray'); axes[0, 0].set_title('Original')
axes[1, 0].imshow(mask, cmap='gray'); axes[1, 0].set_title('Original mask')

for i in range(1, 4):
    aug_i = AugmentationPipeline(aug_cfg, rng=np.random.default_rng(i * 100))
    aug_img, aug_mask = aug_i.apply(img_f32.copy(), mask.copy())
    axes[0, i].imshow(aug_img, cmap='gray')
    axes[0, i].set_title(f'Augmented #{i}')
    axes[1, i].imshow(aug_mask, cmap='gray')
    axes[1, i].set_title(f'Mask #{i}')

for ax in axes.flat: ax.axis('off')
plt.suptitle('Augmentation (applied at training time, never saved)', fontsize=14)
plt.tight_layout(); plt.show()

## 1.6 Generate a Full Dataset (CLI)

Now that you've tested each piece, generate a complete dataset.

### Option A: From the command line

```bash
# Generate 3000 synthetic images (default config)
udm-generate run --config configs/default.yaml

# Generate fewer images for a quick test
udm-generate run --config configs/default.yaml --total 50 --workers 4

# Preview 8 samples without saving to disk
udm-generate preview --n 8 --output /tmp/preview

# Validate a generated dataset
udm-generate validate --output data/synthetic

# Print dataset statistics
udm-generate stats --output data/synthetic
```

### Option B: From Python

In [ ]:
from udm_epic1.dataset.pipeline import DatasetPipeline, PipelineConfig

# Small test run (10 images)
pipe_cfg = PipelineConfig(
    total_images=10,
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2,
    seed=42,
    output_dir='/tmp/synthetic_test',
    num_workers=2,
)

pipeline = DatasetPipeline(pipe_cfg)
manifest_path = pipeline.run()

print(f"\nManifest written to: {manifest_path}")

In [ ]:
# Inspect the generated dataset
import pandas as pd
from pathlib import Path

df = pd.read_csv(manifest_path)
print(f"Total samples: {len(df)}")
print(f"\nSplit distribution:")
print(df['split'].value_counts())
print(f"\nVoid statistics:")
print(f"  Mean voids/image: {df['n_voids'].mean():.1f}")
print(f"  Empty images:     {(df['n_voids'] == 0).sum()}")
print(f"  Mean void area:   {df['total_void_area_fraction'].mean():.4f}")

# Show a sample row
print(f"\nSample row:")
print(df.iloc[0].to_string())

## 1.7 Batch Generation from Your Own Images

If you have a folder of real normal images, generate multiple variants from each.

In [ ]:
def generate_from_real_backgrounds(
    backgrounds_dir: str,
    output_dir: str,
    samples_per_image: int = 5,
    height: int = 512,
    width: int = 512,
    seed: int = 42,
):
    """
    Generate synthetic void images using real normal backgrounds.

    Args:
        backgrounds_dir: folder with your void-free images
        output_dir:      where to save (image, mask) pairs
        samples_per_image: synthetic variants per background
        height, width:   output resolution
        seed:            for reproducibility
    """
    bg_dir = Path(backgrounds_dir)
    out = Path(output_dir)
    (out / 'images').mkdir(parents=True, exist_ok=True)
    (out / 'masks').mkdir(parents=True, exist_ok=True)

    extensions = {'.png', '.tif', '.tiff', '.jpg', '.jpeg', '.bmp'}
    bg_paths = sorted(p for p in bg_dir.iterdir() if p.suffix.lower() in extensions)
    print(f"Found {len(bg_paths)} background images in {bg_dir}")

    cfg = GeneratorConfig(height=height, width=width)
    results = []
    idx = 0

    for bg_path in bg_paths:
        bg = cv2.imread(str(bg_path), cv2.IMREAD_UNCHANGED)
        if bg is None:
            print(f"  Skipping {bg_path.name}")
            continue
        if bg.ndim == 3:
            bg = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)

        for variant in range(samples_per_image):
            gen = SyntheticSampleGenerator(cfg, seed=seed + idx * 1000)
            image_id = f"sample_{idx:06d}"
            img, msk, meta = gen.generate(image_id=image_id, background=bg)

            cv2.imwrite(str(out / 'images' / f'{image_id}.png'), img)
            cv2.imwrite(str(out / 'masks' / f'{image_id}.png'), msk)
            results.append(meta.to_dict())
            idx += 1

    print(f"Generated {idx} samples -> {out}")
    return results

# --- Uncomment to run on your data ---
# results = generate_from_real_backgrounds(
#     backgrounds_dir="data/raw_backgrounds",
#     output_dir="data/synthetic_from_real",
#     samples_per_image=5,
# )

---
# Part 2: Epic 2 — ControlNet-Based Defect Generation

Epic 2 uses **Stable Diffusion + ControlNet** to generate photorealistic defect patches,
then pastes them onto real backgrounds.

The pipeline has 5 stages:
1. **Extract crops** — cut out defect regions from real annotated data
2. **Edge conditioning** — compute Canny edge maps for ControlNet
3. **Train ControlNet** — fine-tune on your defect crops
4. **Generate** — produce new synthetic defects from edge maps
5. **Paste** — blend generated defects onto real backgrounds

**Requirement:** `pip install -e ".[epic2]"` for training/generation (needs GPU).
Crop extraction and pasting work on CPU.

## 2.1 Testing Crop Extraction (`extract_crops_for_pair`)

Given an image + mask pair, extracts one crop per connected component (void).

In [ ]:
# Create a test image with 2 voids
test_img = np.full((200, 200), 150, dtype=np.uint8)
test_mask = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(test_mask, (60, 60), 20, 255, -1)    # void 1
cv2.circle(test_mask, (140, 140), 15, 255, -1)   # void 2

crop_cfg = CropConfig(
    min_component_area_px=50,   # ignore tiny noise
    padding_px=10,              # border around void
    max_crop_side=256,          # max output size
    defect_class='void',
)

crops = list(extract_crops_for_pair(test_img, test_mask, 'test_sample', crop_cfg))
print(f"Extracted {len(crops)} crops")

fig, axes = plt.subplots(len(crops), 3, figsize=(10, 4 * len(crops)))
if len(crops) == 1:
    axes = axes[np.newaxis, :]

for i, (crop_img, crop_mask, meta) in enumerate(crops):
    print(f"  Crop {i}: shape={crop_img.shape}, area={meta['component_area_px']}px, "
          f"bbox=({meta['x0']},{meta['y0']})->({meta['x1']},{meta['y1']})")
    axes[i, 0].imshow(crop_img, cmap='gray'); axes[i, 0].set_title(f'Crop {i} image')
    axes[i, 1].imshow(crop_mask, cmap='gray'); axes[i, 1].set_title(f'Crop {i} mask')
    edge = edge_map_from_mask(crop_mask)
    axes[i, 2].imshow(edge, cmap='gray'); axes[i, 2].set_title(f'Crop {i} edges')

for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

## 2.2 Testing Edge Map Conditioning (`edge_map_from_mask`)

ControlNet uses Canny edge maps as conditioning to guide generation.

In [ ]:
# Test edge detection on different shapes
shapes_for_edges = {
    'circle': lambda m: cv2.circle(m, (64, 64), 30, 255, -1),
    'rectangle': lambda m: cv2.rectangle(m, (30, 30), (100, 100), 255, -1),
    'complex': lambda m: (cv2.circle(m, (50, 50), 20, 255, -1),
                          cv2.circle(m, (80, 80), 15, 255, -1)),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (name, draw_fn) in enumerate(shapes_for_edges.items()):
    m = np.zeros((128, 128), dtype=np.uint8)
    draw_fn(m)
    e = edge_map_from_mask(m)
    axes[0, i].imshow(m, cmap='gray'); axes[0, i].set_title(f'{name} mask')
    axes[1, i].imshow(e, cmap='gray'); axes[1, i].set_title(f'{name} edges')

for ax in axes.flat: ax.axis('off')
plt.suptitle('Edge maps for ControlNet conditioning', fontsize=14)
plt.tight_layout(); plt.show()

## 2.3 Testing Quality Filter (`laplacian_variance`)

After generation, blurry or blank images are rejected using Laplacian variance.

In [ ]:
# Sharp image — should pass
sharp = np.zeros((64, 64), dtype=np.uint8)
sharp[16:48, 16:48] = 200
var_sharp = laplacian_variance(sharp)
ok_sharp, _ = passes_quality_gate(sharp, min_laplacian_var=1.0)

# Blurry image — should fail
blurry = cv2.GaussianBlur(sharp, (31, 31), 10)
var_blurry = laplacian_variance(blurry)
ok_blurry, _ = passes_quality_gate(blurry, min_laplacian_var=1.0)

# Blank image — should fail
blank = np.full((64, 64), 128, dtype=np.uint8)
var_blank = laplacian_variance(blank)
ok_blank, _ = passes_quality_gate(blank, min_laplacian_var=1.0)

print(f"Sharp:  variance={var_sharp:.2f}, passes={ok_sharp}  (expected: True)")
print(f"Blurry: variance={var_blurry:.2f}, passes={ok_blurry}  (expected: False)")
print(f"Blank:  variance={var_blank:.2f}, passes={ok_blank}  (expected: False)")

## 2.4 Testing Paste / Blending (`paste_defect_on_background`)

Paste a defect patch onto a clean background using Poisson or alpha blending.

In [ ]:
# Create a background and a defect patch
bg = np.full((200, 200), 100, dtype=np.uint8)
patch = np.full((60, 60), 200, dtype=np.uint8)
patch_mask = np.zeros((60, 60), dtype=np.uint8)
cv2.circle(patch_mask, (30, 30), 20, 255, -1)

# Alpha blending (works on CPU, always available)
result_alpha = paste_defect_on_background(
    bg, patch, patch_mask,
    center=(100, 100),
    mode='alpha',
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(bg, cmap='gray'); axes[0].set_title('Background')
axes[1].imshow(patch, cmap='gray'); axes[1].set_title('Defect patch')
axes[2].imshow(patch_mask, cmap='gray'); axes[2].set_title('Patch mask')
axes[3].imshow(result_alpha, cmap='gray'); axes[3].set_title('After paste (alpha blend)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print(f"Output shape: {result_alpha.shape}, dtype: {result_alpha.dtype}")

## 2.5 Full Epic 2 Pipeline (CLI)

Here is the complete end-to-end workflow using the CLI.
Run these commands in order in your terminal.

### Step 1: Extract defect crops from your annotated data

```bash
# Input: folder of images + matching masks (same filenames)
udm-epic2 extract-crops \
    --image-dir data/real/train/images \
    --mask-dir  data/real/train/masks \
    --output    data/epic2/crops \
    --config    configs/epic2_dataset.yaml
```

This creates:
```
data/epic2/crops/
    images/void/     <- cropped defect images
    masks/void/      <- cropped masks
    edges/void/      <- Canny edge maps (for ControlNet)
    manifest.csv     <- metadata for all crops
```

### Step 2 (optional): Export to HuggingFace format

```bash
udm-epic2 export-hf \
    --crops-root data/epic2/crops \
    --output     data/epic2/hf_dataset
```

### Step 3: Fine-tune ControlNet (requires GPU)

```bash
# Make sure epic2 deps are installed: uv pip install -e ".[epic2]"
udm-epic2 train --config configs/epic2_train.yaml
```

Key settings in `configs/epic2_train.yaml`:
```yaml
crops_root: data/epic2/crops
output_dir: outputs/epic2_controlnet
max_train_steps: 500         # increase for better quality
learning_rate: 1.0e-5
mixed_precision: fp16        # saves GPU memory
```

### Step 4: Generate synthetic defect patches

```bash
udm-epic2 generate --config configs/epic2_generate.yaml
```

Key settings in `configs/epic2_generate.yaml`:
```yaml
controlnet_path: outputs/epic2_controlnet/controlnet_final
conditioning_glob: data/epic2/crops/edges/void/*.png
output_dir: data/epic2/generated
quality_filter: true          # reject blurry/blank outputs
```

### Step 5: Paste onto real backgrounds

```bash
udm-epic2 paste \
    --background data/real/ok_samples/image_001.png \
    --patch      data/epic2/generated/sample_000.png \
    --mask       data/epic2/crops/masks/void/crop_000.png \
    --center-x 256 --center-y 256 \
    --output     data/epic2/final/augmented_001.png \
    --mode poisson
```

### Step 6: Create ablation template

```bash
# Generate CSV for F1 vs synthetic ratio experiments
udm-epic2 ablation-template --output results/epic2_ablation_template.csv
```

---
# Part 3: End-to-End — From Raw Images to Trained Model

This section ties everything together into a complete workflow.

## Workflow Overview

```
Your real images (no voids)
        │
        ├── Epic 1 path (fast, CPU) ─────────────────────────┐
        │   1. udm-generate run (or background= in Python)   │
        │   2. Output: images/ + masks/ + manifest.csv       │
        │                                                    │
        ├── Epic 2 path (photorealistic, GPU) ───────────────┤
        │   1. udm-epic2 extract-crops (from annotated data) │
        │   2. udm-epic2 train (ControlNet fine-tuning)      │
        │   3. udm-epic2 generate (new defect patches)       │
        │   4. udm-epic2 paste (onto real backgrounds)       │
        │                                                    │
        └── Combined synthetic dataset ──────────────────────┘
                │
                v
        Train segmentation model (YOLOv8, U-Net, etc.)
                │
                v
        Evaluate on real holdout test set
```

## 3.1 Quick Start: Epic 1 Only (No GPU needed)

If you just want synthetic data fast:

```bash
# Step 1: Generate 3000 synthetic images
udm-generate run --config configs/default.yaml

# Step 2: Verify the output
udm-generate validate --output data/synthetic
udm-generate stats --output data/synthetic

# Step 3: Your data is ready at:
#   data/synthetic/train/images/  + data/synthetic/train/masks/
#   data/synthetic/val/images/    + data/synthetic/val/masks/
#   data/synthetic/test/images/   + data/synthetic/test/masks/
#   data/synthetic/manifest.csv
```

To use your own normal images as backgrounds:

```bash
# Step 1: Put your void-free images in data/raw_backgrounds/
# Step 2: Edit configs/default.yaml:
#   image:
#     use_synthetic_backgrounds: false
# Step 3: Run (you'll need to wire the pipeline — see Part 1.7 for Python approach)
```

## 3.2 Full Pipeline: Epic 1 + Epic 2 Combined

For maximum quality, combine both approaches:

```bash
# === EPIC 1: Physics-based baseline ===

# Generate 1000 physics-based synthetic images
udm-generate run --config configs/default.yaml --total 1000


# === EPIC 2: ControlNet photorealistic ===

# Prep: extract crops from your real annotated data
udm-epic2 extract-crops \
    --image-dir data/real/train/images \
    --mask-dir  data/real/train/masks \
    --output    data/epic2/crops \
    --config    configs/epic2_dataset.yaml

# Train ControlNet (GPU required)
udm-epic2 train --config configs/epic2_train.yaml

# Generate 1000 ControlNet defect patches
udm-epic2 generate --config configs/epic2_generate.yaml

# Paste generated patches onto real backgrounds
# (loop over your backgrounds — see script below)


# === COMBINE: Merge both datasets ===
# Your training set now has:
#   - Real annotated images
#   - Epic 1 physics-based synthetic
#   - Epic 2 ControlNet synthetic


# === EVALUATE: Ablation study ===
udm-epic2 ablation-template --output results/ablation.csv
# Fill in F1 scores for: 0%, 25%, 50%, 75%, 100% synthetic ratios
```

## 3.3 Batch Paste Script (Epic 2)

The CLI `paste` command handles one image at a time.
Here's how to do it in batch:

In [ ]:
def batch_paste(
    background_dir: str,
    patch_dir: str,
    mask_dir: str,
    output_dir: str,
    mode: str = 'alpha',
    seed: int = 42,
):
    """
    Paste generated defect patches onto multiple backgrounds.

    For each background, picks a random patch and pastes it
    at a random location.
    """
    from udm_epic2.integration.paste import paste_defect_on_background

    bg_dir = Path(background_dir)
    p_dir = Path(patch_dir)
    m_dir = Path(mask_dir)
    out = Path(output_dir)
    (out / 'images').mkdir(parents=True, exist_ok=True)
    (out / 'masks').mkdir(parents=True, exist_ok=True)

    bg_paths = sorted(bg_dir.glob('*.png'))
    patch_paths = sorted(p_dir.glob('*.png'))
    mask_paths = sorted(m_dir.glob('*.png'))
    rng = np.random.default_rng(seed)

    for i, bg_path in enumerate(bg_paths):
        bg = cv2.imread(str(bg_path), cv2.IMREAD_UNCHANGED)
        if bg is None:
            continue

        # Pick a random patch
        idx = rng.integers(0, len(patch_paths))
        patch = cv2.imread(str(patch_paths[idx]), cv2.IMREAD_UNCHANGED)
        mask = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)

        # Random paste location (within bounds)
        ph, pw = patch.shape[:2]
        cx = int(rng.integers(pw // 2, bg.shape[1] - pw // 2))
        cy = int(rng.integers(ph // 2, bg.shape[0] - ph // 2))

        result = paste_defect_on_background(bg, patch, mask, (cx, cy), mode=mode)

        cv2.imwrite(str(out / 'images' / f'aug_{i:06d}.png'), result)
        # The mask for the pasted region
        full_mask = np.zeros(bg.shape[:2], dtype=np.uint8)
        y0 = max(0, cy - ph // 2)
        x0 = max(0, cx - pw // 2)
        y1 = min(bg.shape[0], y0 + ph)
        x1 = min(bg.shape[1], x0 + pw)
        mh, mw = y1 - y0, x1 - x0
        full_mask[y0:y1, x0:x1] = mask[:mh, :mw]
        cv2.imwrite(str(out / 'masks' / f'aug_{i:06d}.png'), full_mask)

    print(f"Pasted {len(bg_paths)} images -> {out}")

# --- Uncomment to run ---
# batch_paste(
#     background_dir='data/real/ok_samples',
#     patch_dir='data/epic2/generated',
#     mask_dir='data/epic2/crops/masks/void',
#     output_dir='data/epic2/final',
# )

---
# Part 4: Running Tests

Before using the repo, run the test suite to verify everything works:

```bash
# Run all tests with coverage
pytest tests/ -v

# Run only Epic 1 tests
pytest tests/test_epic1.py -v

# Run only Epic 2 tests
pytest tests/test_epic2.py -v
```

### What the tests verify

**Epic 1 (`test_epic1.py`):**
- Physics: Beer-Lambert correctness, void brightness > background
- Void shapes: all 4 morphologies produce valid masks
- Generator: output dtypes, reproducibility, metadata, empty image fraction
- Augmentation: shape preservation, range clipping, ring artifacts

**Epic 2 (`test_epic2.py`):**
- Edge maps: Canny detection on empty/filled masks
- Crop extraction: single component, manifest writing
- Paste: alpha blending output shape
- Quality filter: Laplacian variance calculation
- HF export: metadata CSV + folder structure

---
# Part 5: Configuration Reference

## Config files

| File | Epic | Purpose |
|------|------|---------|
| `configs/default.yaml` | 1 | Full dataset generation (physics, voids, augmentation) |
| `configs/epic2_dataset.yaml` | 2 | Crop extraction settings |
| `configs/epic2_train.yaml` | 2 | ControlNet training hyperparameters |
| `configs/epic2_generate.yaml` | 2 | Generation settings (model path, quality filter) |

## Key parameters to tune

### Epic 1 (`configs/default.yaml`)

| Parameter | Default | What it does |
|-----------|---------|-------------|
| `dataset.total_images` | 3000 | Number of images to generate |
| `image.use_synthetic_backgrounds` | true | false = use your own images from `backgrounds_dir` |
| `voids.count_range` | [0, 8] | Min/max voids per image |
| `voids.min_area_fraction` | 0.001 | Smallest void (0.1% of image) |
| `voids.max_area_fraction` | 0.25 | Largest void (25% of image) |
| `voids.empty_image_fraction` | 0.15 | Fraction with 0 voids (hard negatives) |
| `physics.apply_sft_correction` | true | Set false if your images are already corrected |

### Epic 2 (`configs/epic2_train.yaml`)

| Parameter | Default | What it does |
|-----------|---------|-------------|
| `max_train_steps` | 500 | More steps = better quality (try 1000-5000) |
| `learning_rate` | 1e-5 | Lower if training is unstable |
| `mixed_precision` | fp16 | Use bf16 on A100/H100 GPUs |
| `resolution` | 512 | Match your crop sizes |
| `checkpointing_steps` | 250 | Save intermediate checkpoints |

---
# Part 6: CLI Command Reference

## Epic 1: `udm-generate`

| Command | Description |
|---------|------------|
| `udm-generate run` | Generate full dataset |
| `udm-generate preview --n 8` | Generate N samples for visual inspection |
| `udm-generate validate --output data/synthetic` | Check dataset integrity |
| `udm-generate stats --output data/synthetic` | Print manifest statistics |

## Epic 2: `udm-epic2`

| Command | Description |
|---------|------------|
| `udm-epic2 extract-crops` | Extract defect crops from image+mask pairs |
| `udm-epic2 export-hf` | Convert crops to HuggingFace folder layout |
| `udm-epic2 train` | Fine-tune ControlNet on defect crops |
| `udm-epic2 generate` | Generate new defect patches from edge maps |
| `udm-epic2 paste` | Blend defect patch onto background |
| `udm-epic2 ablation-template` | Create CSV for synthetic ratio experiments |

---
# What's Next?

Based on the project roadmap, the next epics are:

| Priority | Epic | Description | Status |
|----------|------|-------------|--------|
| **P0** | Epic 4: DANN | Domain adaptation across 4 manufacturing sites | TODO |
| **P1** | Epic 5: Active Learning | Smart sample selection — 90% performance with 10% labels | TODO (needs Epic 4) |
| **P2** | Epic 3: CycleGAN | Cross-modality translation (AOI to USM) | TODO |

Epic 2 and Epic 3 can run in parallel. Epic 4 (DANN) is the **critical path** — highest business impact.

See the [Epic Roadmap](https://www.notion.so/303ea36d4bc081d18f38c699bda715b0) in Notion for full details.